# ira-esm-tokenizer on Kaggle

Everything in one notebook. Build the training set, train the tokenizer, then
ask whether the tokens it produces actually mean anything.

Source code comes from the attached **esm data** dataset rather than being
written out inline, so the notebook stays readable and the code stays
versioned in one place.

**Before you run anything**, set these in the right-hand sidebar:

- **Accelerator** -> GPU T4 x2
- **Internet** -> On  (needed for pip and for downloading PDB files)
- **Add Input** -> attach the `esm data` dataset

Changing the accelerator or internet setting restarts the kernel and wipes all
state, so set those first, then Run All.


## 1. Check the session


In [ ]:
import torch, subprocess

print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                          "--format=csv,noheader"], capture_output=True, text=True).stdout.strip())
else:
    print("NO GPU. Set Accelerator to GPU T4 x2 in the sidebar (this restarts the kernel).")

# Internet check -- if this fails, flip Internet on in the sidebar.
import urllib.request
try:
    urllib.request.urlopen("https://files.rcsb.org", timeout=10)
    print("internet: ok")
except Exception as e:
    print("internet: OFF ->", e)

## 2. Install dependencies


In [ ]:
!pip install -q biopython
print('done')

## 3. Bring in the source files

The `esm data` dataset holds the project's `.py` files. This copies them into
`/kaggle/working/ira-esm-tokenizer`, which is writable and survives a
*Save Version*, so checkpoints land in Kaggle's output.

Kaggle sometimes auto-extracts an uploaded `.zip` and sometimes leaves it
alone, so this handles both cases and tells you what is actually attached if
it finds neither.


In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")                      # datasets, mounted read-only
ROOT  = Path("/kaggle/working/ira-esm-tokenizer")  # where we want the project, writable

# Kaggle sometimes auto-extracts a .zip when it builds a Dataset and sometimes
# leaves it alone, so handle both. The ", None" matters. Without a default,
# next() raises StopIteration instead of letting us fall through.
zip_path = next(INPUT.rglob("*.zip"), None)

if zip_path is not None:
    # The zip already contains a top-level "ira-esm-tokenizer/" folder, so
    # extract to ROOT's PARENT and the contents land at ROOT rather than one
    # level too deep.
    zipfile.ZipFile(zip_path).extractall(ROOT.parent)
    print("unzipped", zip_path.name)
else:
    # Already extracted. Rather than guess the folder name, which changes with
    # the dataset name and is sometimes flattened away, find train.py and treat
    # wherever it sits as the project root.
    train_py = next(INPUT.rglob("train.py"), None)
    if train_py is None:
        print("No code found under /kaggle/input. What is actually attached:")
        for p in sorted(INPUT.rglob("*"))[:40]:
            print("   ", p)
        raise SystemExit("Attach the 'esm data' dataset in the sidebar, then re-run.")
    # /kaggle/input is read-only, so copy into the writable working dir.
    # dirs_exist_ok lets this cell be re-run without blowing up.
    shutil.copytree(train_py.parent, ROOT, dirs_exist_ok=True)
    print("copied from", train_py.parent)

# chdir because the scripts use relative paths like "data/parsed", and
# sys.path.insert because "from model.encoder import ..." searches sys.path,
# which does NOT include the current directory inside a notebook.
os.chdir(ROOT); sys.path.insert(0, str(ROOT))

# Expect 11 files. Fewer means the upload was incomplete, so read the list.
print("ready:", sorted(p.name for p in ROOT.rglob("*.py")))

## 4. Verify the padding fix landed

`collate_fn` pads with zero coordinates. Without the clamps in
`build_frames`, that produces NaN which spreads into the *real* residues too,
and every batch containing padding goes to NaN loss. This cell proves the
loaded code is the fixed version before you spend GPU hours on it.

In [ ]:
import torch, importlib
import model.geometry, model.encoder, model.quantizer, model.decoder
for m in [model.geometry, model.encoder, model.quantizer, model.decoder]:
    importlib.reload(m)
from model.encoder import StructureEncoder
from model.quantizer import VectorQuantizer
from model.decoder import StructureDecoder, fape_loss

torch.manual_seed(0)
B, L = 2, 12
coords = torch.randn(B, L, 4, 3) * 5
mask = torch.ones(B, L, dtype=torch.bool)
coords[1, 7:] = 0.0        # exactly what collate_fn produces for padding
mask[1, 7:] = False

enc, vq, dec = StructureEncoder(), VectorQuantizer(), StructureDecoder()
z = enc(coords, mask); q = vq(z, mask); out = dec(q["quantized"], mask)
loss = fape_loss(out, coords, mask) + q["loss"]

assert torch.isfinite(loss), "NaN loss -- geometry.py is missing the clamp fix"
assert not torch.isnan(z[mask]).any(), "NaN leaked into real residues"
print(f"padded-batch loss = {loss.item():.4f}  (finite, good)")

## 5. Build the training set

Three steps: ask RCSB which structures to use, download them, parse them into
`.npz` coordinate arrays. All three skip work that's already done, so
re-running this cell after a restart is cheap.

This takes a while the first time. See section 6 for how to avoid ever
repeating it.

In [ ]:
import subprocess, sys
from pathlib import Path

def run(*args):
    """Run a pipeline step and stop loudly if it fails, instead of letting
    a silent failure look like an empty dataset three cells later."""
    print(">", " ".join(args), flush=True)
    result = subprocess.run([sys.executable, *args])
    if result.returncode != 0:
        raise SystemExit(f"step failed: {' '.join(args)}")

# Bump --max-results once the pipeline is proven. 1000 structures is enough
# to get a real signal and small enough to download in reasonable time.
if not Path("data/pdb_ids.txt").exists():
    run("data/fetch_pdb_ids.py", "--max-results", "1000",
        "--min-length", "50", "--max-length", "300")
else:
    print("data/pdb_ids.txt already exists, skipping the RCSB query")

run("data/download_pdbs.py", "--id-list", "data/pdb_ids.txt", "--out-dir", "data/raw_pdb")
run("data/parse_structures.py", "--pdb-dir", "data/raw_pdb",
    "--out-dir", "data/parsed", "--min-length", "50")

n = len(list(Path("data/parsed").glob("*.npz")))
print(f"\nparsed structures ready: {n}")
assert n > 0, "nothing parsed -- check the download output above"

## 6. Save the parsed data so you never rebuild it

Downloading a thousand PDB files every session is a waste of your GPU quota.
Run this once, then use *File -> Save Version* so `/kaggle/working` is kept.
After that, create a Kaggle Dataset from the output and attach it to the
notebook. On later runs, skip section 5 and run this instead.

In [ ]:
from pathlib import Path
import shutil

# If you've attached the parsed data as a Kaggle Dataset, point this at it.
# Adjust the folder name to whatever you called the dataset.
ATTACHED = Path("/kaggle/input/ira-esm-parsed")

if ATTACHED.exists():
    target = Path("data/parsed"); target.mkdir(parents=True, exist_ok=True)
    for f in ATTACHED.glob("*.npz"):
        if not (target / f.name).exists():
            shutil.copy(f, target / f.name)
    print("copied", len(list(target.glob('*.npz'))), "structures from the attached dataset")
else:
    print("no attached dataset found at", ATTACHED)
    print("using whatever section 5 built:",
          len(list(Path('data/parsed').glob('*.npz'))), "structures")

## 7. Train

Run in the background with `nohup` so a dropped browser connection doesn't
kill the run, and tail the log to watch it.

`--budget` is the memory control: batches are filled up to
(count x longest-protein-squared), because the encoder's cost grows with the
square of the length. 262144 is about 4 proteins of 256 residues at a time.
A single T4 handles that comfortably. Halve it if you hit out-of-memory,
double it if `nvidia-smi` shows the card idling.

Only one GPU is used. The model is small and the second T4 would cost more in
synchronisation than it returns.

In [ ]:
import subprocess, os

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)

cmd = 'nohup python -u train.py \\\n  --parsed-dir data/parsed \\\n  --checkpoint-dir /kaggle/working/checkpoints \\\n  --epochs 200 \\\n  --max-length 256 \\\n  --budget 262144 \\\n  --lr 3e-4 \\\n  --num-workers 2 \\\n  --revive-every 200 \\\n  > /kaggle/working/train.log 2>&1 &'

env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")
subprocess.Popen(cmd, shell=True, env=env, cwd="/kaggle/working/ira-esm-tokenizer")
print("training started -- run the next cell to watch it")

In [ ]:
# Watch progress. Re-run this cell whenever you want an update.
!tail -n 25 /kaggle/working/train.log

### Reading the output

```
epoch  12  train 6.412A  val 6.388A  vq 0.0431  codes 3480/4096  ppl 2611  revived 4  38s
```

- **train / val** -- reconstruction error in Angstroms. The number to watch.
  It is a clamped average so it can never exceed 10, and it starts near 10.
- **vq** -- how far encoder outputs sit from their assigned codes. Should
  fall fast and stay small. If it stays large, the codebook is chasing the
  encoder rather than tracking it.
- **codes / ppl** -- how much of the codebook is genuinely in use.
  `codes` counts anything used at least once; `ppl` (perplexity) is the
  honest version. If 3000 codes fire once each and 20 absorb everything,
  `codes` says 3000 and `ppl` says about 20. Watch `ppl`.
- **revived** -- dead codes reset onto real data this epoch. High early,
  should settle toward zero.

## 8. Resuming after a session ends

Kaggle stops sessions on a time limit, and the GPU quota is weekly. The run
checkpoints every epoch, so pick up where it stopped:

1. *File -> Save Version* before the session ends, so `/kaggle/working` persists.
2. On the new session, run sections 1-4 and 6, then this cell.

In [ ]:
import subprocess, os
from pathlib import Path

ckpt = Path("/kaggle/working/checkpoints/last.pt")
assert ckpt.exists(), "no checkpoint found -- start a fresh run from section 7"

import torch
state = torch.load(ckpt, map_location="cpu")
print(f"resuming from epoch {state['epoch']}, best val so far {state['best_val']:.3f}A")

cmd = f'nohup python -u train.py \\\n  --parsed-dir data/parsed \\\n  --checkpoint-dir /kaggle/working/checkpoints \\\n  --resume /kaggle/working/checkpoints/last.pt \\\n  --epochs 200 --max-length 256 --budget 262144 --lr 3e-4 \\\n  --num-workers 2 --revive-every 200 \\\n  >> /kaggle/working/train.log 2>&1 &'
subprocess.Popen(cmd, shell=True, env=dict(os.environ, CUDA_VISIBLE_DEVICES="0"),
                 cwd="/kaggle/working/ira-esm-tokenizer")
print("resumed")

## 9. Plot the curves

In [ ]:
import re
import matplotlib.pyplot as plt

rows = []
for line in open("/kaggle/working/train.log"):
    m = re.match(r"epoch\s+(\d+)\s+train ([\d.]+)A\s+val ([\d.]+)A\s+vq ([\d.]+)\s+"
                 r"codes (\d+)/\d+\s+ppl (\d+)", line)
    if m:
        rows.append([float(x) for x in m.groups()])

assert rows, "no epoch lines in the log yet"
epoch, train, val, vq, codes, ppl = zip(*rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(epoch, train, label="train"); axes[0].plot(epoch, val, label="val")
axes[0].set_title("reconstruction error"); axes[0].set_xlabel("epoch")
axes[0].set_ylabel("Angstroms"); axes[0].legend()

axes[1].plot(epoch, vq, color="tab:red"); axes[1].set_yscale("log")
axes[1].set_title("VQ loss (log scale)"); axes[1].set_xlabel("epoch")

axes[2].plot(epoch, codes, label="codes used")
axes[2].plot(epoch, ppl, label="perplexity")
axes[2].set_title("codebook usage"); axes[2].set_xlabel("epoch"); axes[2].legend()

plt.tight_layout(); plt.show()
print(f"best val: {min(val):.3f}A at epoch {int(epoch[val.index(min(val))])}")

## 10. Token quality, and how this differs from ESM-3
Reconstruction error only shows that the decoder can rebuild a backbone from
the tokens. It does not show that the tokens mean anything, so the next two
cells go and check.

**`analyze_codebook.py`** asks three questions of a trained codebook.

1. Do codes line up with secondary structure? A code picked almost only on
   helices has clearly learned "helix-ish".
2. Do codes line up with local backbone geometry? Secondary structure is a
   coarse 3-way label, and local geometry is the continuous thing the encoder
   actually sees, so this is the finer-grained version.
3. Is the mapping consistent across proteins? This is the real test. The same
   local shape in two different proteins should get the same code. Where it
   does not, the token is partly describing which protein it came from rather
   than what shape it is.

**`compare_esm3.py`** turns the architecture differences into measurements.

| | ESM-3 | here |
|---|---|---|
| receptive field | 16 nearest neighbours | whole chain |
| position information in the encoder | relative positional embedding | none at all |
| codebook update | moving average (EMA) | gradient |
| training objective | reconstruction plus distance, error and confidence heads | reconstruction alone |
| decoder | transformer with 6D rotation frames | close match |

Only the first two of those can be probed from a checkpoint, since they are
properties of the forward pass. So the script re-runs the trained encoder with
its attention clipped to each residue's 16 nearest neighbours, and separately
with the residues shuffled, and reports how far the tokens move. The other
three are properties of training, so what it reports there is their
consequence, which is the shape of the codebook we ended up with.


In [ ]:
# ---------------------------------------------------------------------------
# Both scripts read a checkpoint and neither trains, so this is safe to re-run
# at any point once section 7 has produced at least one epoch.
# ---------------------------------------------------------------------------
import subprocess, sys
from pathlib import Path

CKPT = Path("/kaggle/working/checkpoints/best.pt")
if not CKPT.exists():
    CKPT = Path("/kaggle/working/checkpoints/last.pt")
assert CKPT.exists(), "no checkpoint yet -- train first (section 7)"

def run(script, *extra):
    # check=False, not check=True. If one analysis fails we still want the
    # other one's output rather than an empty cell.
    print("\n" + "#" * 70); print("#", script); print("#" * 70, flush=True)
    subprocess.run([sys.executable, script,
                    "--checkpoint", str(CKPT),
                    "--parsed-dir", "data/parsed",
                    "--out-dir", "/kaggle/working/analysis", *extra], check=False)

run("analyze_codebook.py", "--max-structures", "200", "--plots")
run("compare_esm3.py", "--max-structures", "100")

In [ ]:
# The three figures analyze_codebook.py --plots wrote. The first is the one to
# look at. If the codebook learned geometry rather than just a convenient
# partition of its own feature space, the helix codes and the strand codes
# land in visibly different regions.
from pathlib import Path
from IPython.display import Image, display

for name in ["code_geometry_map.png", "codebook_usage.png", "code_purity.png"]:
    p = Path("/kaggle/working/analysis") / name
    if p.exists():
        print(name)
        display(Image(filename=str(p)))
    else:
        print("missing:", name)